<a href="https://colab.research.google.com/github/Navendu13/apriori-alpha-project/blob/main/apriori_game_theoretic_alpha_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q yfinance

## Step 1: Data Acquisition — Building the Research Universe

**What this step does:**
This cell downloads 11+ years (Jan 2015 – Jul 2026) of daily OHLCV (Open, High, Low, Close, Volume)
data for a 30-stock cross-sector universe using the `yfinance` API, then extracts and saves clean
close price and volume matrices as CSV files.

**Why we are doing this:**
Every downstream step (signal construction, hypothesis testing, backtesting) depends entirely on
having clean, aligned price and volume data. Without a properly structured dataset, no signal or
statistical test can be trusted.

**Why this specific universe (30 stocks, cross-sector):**
We deliberately chose a diversified basket spanning Tech, Financials, Healthcare, Energy, and
Consumer sectors rather than a single sector. This is a methodological safeguard: if a signal only
works in one sector, it's likely a sector-specific fluke rather than a genuine market microstructure
effect. A signal that holds across sectors is far more credible and defensible.

**Why this specific timeframe (2015–2026):**
This window intentionally spans multiple market regimes — the 2015-2019 low-volatility bull run,
the 2020 COVID crash and recovery, the 2022 rate-hike bear market, and the 2023-2026 recovery.
Testing across regimes is a standard robustness check; a signal that only works in one type of
market environment is not reliable.

**What happens if we skip or shortcut this step:**
Any signal or backtest built on incomplete, misaligned, or narrow data (e.g., only bull-market years,
or only tech stocks) risks being overfit or non-generalizable — a critical flaw that any technically
rigorous reviewer (like a quant fund CTO) would immediately flag.

In [ ]:
import yfinance as yf
import pandas as pd, os

# Diversified, liquid, cross-sector universe — avoids single-sector overfit criticism
universe = [
    'AAPL','MSFT','NVDA','GOOGL','AMZN','META','AVGO','TSLA',   # Tech/Comm
    'JPM','BAC','GS','MS',                                       # Financials
    'UNH','JNJ','PFE','LLY',                                     # Healthcare
    'XOM','CVX','COP',                                           # Energy
    'PG','KO','PEP','WMT','COST',                                # Consumer staples
    'HD','DIS','NFLX','ADBE','CRM','V'                           # Consumer disc./services
]

# 2015–2026 spans multiple regimes: 2015-19 bull, 2020 COVID crash/recovery,
# 2022 rate-hike bear market, 2023-26 recovery — needed for walk-forward robustness
start_date = '2015-01-01'
end_date = '2026-07-29'

data = yf.download(universe, start=start_date, end=end_date,
                    group_by='ticker', auto_adjust=True, progress=False)

os.makedirs('output', exist_ok=True)

close_prices = pd.concat({t: data[t]['Close'] for t in universe
                           if t in data.columns.get_level_values(0)}, axis=1)
volumes = pd.concat({t: data[t]['Volume'] for t in universe
                      if t in data.columns.get_level_values(0)}, axis=1)

close_prices.to_csv('output/close_prices.csv')
volumes.to_csv('output/volumes.csv')

print(close_prices.shape, volumes.shape)
close_prices.tail()

(2908, 30) (2908, 30)


,AAPL,MSFT,NVDA,GOOGL,AMZN,META,AVGO,TSLA,JPM,BAC,...,KO,PEP,WMT,COST,HD,DIS,NFLX,ADBE,CRM,V
Date,,,,,,,,,,,,,,,,,,,,,
2026-07-22,325.890015,390.339996,212.059998,342.089996,244.850006,627.169983,396.809998,374.010010,348.209991,61.619999,...,82.199997,135.649994,109.330002,925.838013,331.450012,95.870003,68.529999,218.360001,163.000000,353.420013
2026-07-23,321.660004,381.579987,208.759995,317.690002,233.660004,606.099976,392.470001,319.690002,349.899994,61.279999,...,81.169998,134.949997,108.400002,924.589966,324.709991,92.830002,68.889999,212.169998,156.929993,351.600006
2026-07-24,333.019989,381.700012,206.839996,319.739990,232.110001,595.190002,381.920013,313.029999,353.209991,62.049999,...,82.250000,136.639999,109.470001,935.030029,332.980011,94.849998,70.089996,225.110001,163.660004,355.739990
2026-07-27,336.910004,389.100006,196.509995,326.559998,231.389999,593.869995,383.220001,309.220001,356.200012,62.130001,...,84.070000,139.789993,111.739998,951.580017,336.089996,96.650002,70.400002,237.750000,173.600006,362.529999
2026-07-28,340.079987,393.350006,197.009995,333.709991,230.860001,593.409973,380.910004,307.440002,357.309998,62.619999,...,88.269997,142.860001,113.099998,966.580017,344.470001,98.889999,72.389999,249.179993,181.500000,366.589996


## Step 1 Results: Data Validation

**What the output shows:**
The pull returned a (2908, 30) matrix for both close prices and volumes — meaning 2,908 trading
days across all 30 tickers, with no missing tickers, confirming a complete and aligned dataset.

**Interpreting the numbers:**
2,908 trading days is consistent with ~11.5 years of data (roughly 252 trading days/year), which
matches our intended 2015–2026 window. The tail rows show plausible, realistic closing prices
(e.g., AAPL ~$340, NVDA ~$197) with no NaNs or broken values visible.

**Is this realistic enough to proceed:**
Yes. Matching row/column counts across both price and volume matrices confirms no silent data
loss or misalignment occurred during the download and merge process.

**How this feeds into the next step:**
This clean price/volume matrix is the direct input for VPIN signal construction (Step 2). Any
gaps or misalignment here would propagate errors into every later calculation, so validating
shape and sanity now prevents having to debug much more confusing errors later in the pipeline.

## Step 2: Constructing the VPIN Signal (Game-Theoretic Order Flow Proxy)

**What this step does:**
This cell computes VPIN (Volume-Synchronized Probability of Informed Trading) for each stock,
using daily returns and volume as inputs. It classifies each day's volume into estimated
buy-initiated vs. sell-initiated portions using a z-score-based Bulk Volume Classification (BVC)
method, then computes a rolling order-flow imbalance ratio.

**Why we are doing this:**
VPIN is grounded in the Easley-O'Hara sequential trade model, a game-theoretic framework where
market makers (uninformed) and informed traders interact strategically. Market makers cannot
directly observe who is informed, so they infer risk from the imbalance of buy vs. sell order
flow. This directly aligns with A Priori's stated focus on "game theory" — it is not a generic
technical indicator, but a signal rooted in strategic market microstructure theory.

**Why we approximate with BVC instead of true tick-level classification:**
The gold-standard method (Lee-Ready algorithm) requires tick-by-tick trade and quote data, which
we don't have. BVC is the accepted academic substitute when only daily OHLCV is available — it
infers buy/sell pressure from how much price moved relative to recent volatility.

**What happens if we skip this step:**
Without VPIN (or an equivalent microstructure-based proxy), the project would just be a standard
price/volume technical analysis exercise, with no genuine connection to game theory — undermining
the core differentiation strategy for this pitch.

In [ ]:
import pandas as pd, numpy as np, os

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vol = pd.read_csv('output/volumes.csv', index_col=0, parse_dates=True)

def compute_vpin(price, volume, bucket_size_frac=0.02, window=50):
    ret = price.pct_change()
    daily_vol = volume
    total_vol = daily_vol.sum()
    bucket_size = total_vol * bucket_size_frac
    sigma = ret.rolling(20).std()
    z = (ret / sigma).clip(-5, 5)
    from scipy.stats import norm
    buy_frac = norm.cdf(z.fillna(0))
    buy_vol = daily_vol * buy_frac
    sell_vol = daily_vol * (1 - buy_frac)
    imbalance = (buy_vol - sell_vol).abs()
    vpin = imbalance.rolling(window).sum() / daily_vol.rolling(window).sum()
    return vpin

vpin_df = pd.DataFrame(index=close.index)
for t in close.columns:
    vpin_df[t] = compute_vpin(close[t], vol[t])

vpin_df.to_csv('output/vpin_signal.csv')
print(vpin_df.shape)
print(vpin_df.tail(3))
print(vpin_df.describe().T[['mean','std','min','max']].head(10))

(2908, 30)
                AAPL      MSFT      NVDA     GOOGL      AMZN      META  \
Date                                                                     
2026-07-24  0.550279  0.572202  0.515895  0.515387  0.579539  0.529870   
2026-07-27  0.547598  0.579197  0.521924  0.510596  0.569668  0.521824   
2026-07-28  0.550328  0.579532  0.504956  0.517625  0.561183  0.520117   

                AVGO      TSLA       JPM       BAC  ...        KO       PEP  \
Date                                                ...                       
2026-07-24  0.551437  0.557421  0.542339  0.495780  ...  0.535317  0.520605   
2026-07-27  0.549724  0.543886  0.538618  0.481685  ...  0.545904  0.521390   
2026-07-28  0.538352  0.544077  0.541207  0.490725  ...  0.567102  0.529319   

                 WMT      COST        HD       DIS      NFLX      ADBE  \
Date                                                                     
2026-07-24  0.496395  0.532467  0.531131  0.500173  0.517117  0.554927   


## Step 2 Results: VPIN Signal Validation

**What the output shows:**
VPIN values across all 30 stocks average around 0.51–0.53, with standard deviations of roughly
0.04–0.05, and a full range spanning approximately 0.20 to 0.76.

**Interpreting the numbers:**
VPIN is bounded between 0 and 1 by construction. A value near 0.5 indicates balanced buy/sell
volume (normal, uninformed trading conditions). Values pushing toward 0.6–0.76 indicate periods
where volume is heavily skewed to one side — a signal of potentially elevated informed trading
or order flow "toxicity."

**Is this realistic enough to proceed:**
Yes. This distribution (centered near 0.5, with moderate spread and occasional extremes) is
consistent with published VPIN studies in the market microstructure literature. If our values had
clustered near 0 or 1, or shown no variation at all, that would indicate a bug in the calculation.

**How this feeds into the next step:**
At this stage, VPIN is just a number sitting next to price history — it has no proven predictive
value yet. Step 3 formally tests whether elevated VPIN actually precedes unusual returns or
volatility, which determines whether this signal is a real, usable input or just noise.

## Step 3: Hypothesis Testing — Does VPIN Actually Predict Anything?

**What this step does:**
This cell tests whether high-VPIN days are followed by statistically different forward returns
(1-day, 5-day) and forward volatility (5-day) compared to low-VPIN days. For each stock, VPIN is
ranked into quintiles based on its own historical distribution, and a Welch's t-test compares the
extreme (top vs. bottom) quintiles.

**Why we are doing this:**
A signal is only useful if it has demonstrated, statistically significant predictive power —
otherwise it's just an interesting-looking number with no practical value. This step is the
critical "does it actually work" checkpoint before any strategy is built on top of VPIN.

**Why we shift returns/volatility forward (no lookahead bias):**
Forward returns and volatility are computed using `.shift(-1)` and `.shift(-5)`, ensuring VPIN on
day t is only ever compared to information that occurs strictly after day t. This prevents
lookahead bias — a critical, commonly-tested error in quant interviews, where future information
accidentally leaks into a predictive signal, producing falsely inflated results.

**Why we test across all 30 stocks instead of just one:**
Testing a signal on a single stock risks finding a coincidental pattern (overfitting/noise).
Requiring consistent results across a diversified 30-stock universe is a standard robustness
check that distinguishes a genuine market effect from a fluke.

**What happens if we skip this step:**
Building a trading strategy directly on VPIN without this validation step would risk deploying
a signal that is pure noise, dressed up as insight — exactly the kind of unvalidated claim that a
quant reviewer would immediately reject.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)

fwd_ret_1d = close.pct_change().shift(-1)
fwd_ret_5d = close.pct_change(5).shift(-5)
fwd_vol_5d = close.pct_change().rolling(5).std().shift(-5)

results = []
for t in close.columns:
    df = pd.DataFrame({
        'vpin': vpin[t],
        'fwd_ret_1d': fwd_ret_1d[t],
        'fwd_ret_5d': fwd_ret_5d[t],
        'fwd_vol_5d': fwd_vol_5d[t]
    }).dropna()
    if len(df) < 200:
        continue
    df['quintile'] = pd.qcut(df['vpin'], 5, labels=False, duplicates='drop')
    low = df[df['quintile'] == 0]
    high = df[df['quintile'] == df['quintile'].max()]
    t_ret1, p_ret1 = stats.ttest_ind(high['fwd_ret_1d'], low['fwd_ret_1d'], equal_var=False)
    t_ret5, p_ret5 = stats.ttest_ind(high['fwd_ret_5d'], low['fwd_ret_5d'], equal_var=False)
    t_vol5, p_vol5 = stats.ttest_ind(high['fwd_vol_5d'], low['fwd_vol_5d'], equal_var=False)
    results.append({
        'ticker': t,
        'low_vpin_fwdret1d': low['fwd_ret_1d'].mean(),
        'high_vpin_fwdret1d': high['fwd_ret_1d'].mean(),
        'p_ret1d': p_ret1,
        'low_vpin_fwdret5d': low['fwd_ret_5d'].mean(),
        'high_vpin_fwdret5d': high['fwd_ret_5d'].mean(),
        'p_ret5d': p_ret5,
        'low_vpin_fwdvol5d': low['fwd_vol_5d'].mean(),
        'high_vpin_fwdvol5d': high['fwd_vol_5d'].mean(),
        'p_vol5d': p_vol5,
    })

res_df = pd.DataFrame(results)
res_df.to_csv('output/vpin_hypothesis_test.csv', index=False)

print("=== Volatility prediction (high VPIN vs low VPIN, 5-day forward vol) ===")
print(f"Stocks where high VPIN -> higher fwd vol: {(res_df['high_vpin_fwdvol5d'] > res_df['low_vpin_fwdvol5d']).sum()} / {len(res_df)}")
print(f"Stocks with significant vol diff (p<0.05): {(res_df['p_vol5d'] < 0.05).sum()} / {len(res_df)}")
print()
print("=== Return prediction (5-day) ===")
print(f"Stocks with significant return diff (p<0.05): {(res_df['p_ret5d'] < 0.05).sum()} / {len(res_df)}")
print()
print(res_df[['ticker','low_vpin_fwdvol5d','high_vpin_fwdvol5d','p_vol5d','p_ret5d']].round(4).to_string(index=False))

=== Volatility prediction (high VPIN vs low VPIN, 5-day forward vol) ===
Stocks where high VPIN -> higher fwd vol: 30 / 30
Stocks with significant vol diff (p<0.05): 28 / 30

=== Return prediction (5-day) ===
Stocks with significant return diff (p<0.05): 7 / 30

ticker  low_vpin_fwdvol5d  high_vpin_fwdvol5d  p_vol5d  p_ret5d
  AAPL             0.0119              0.0203   0.0000   0.0009
  MSFT             0.0129              0.0181   0.0000   0.1512
  NVDA             0.0228              0.0318   0.0000   0.0045
 GOOGL             0.0139              0.0199   0.0000   0.7272
  AMZN             0.0164              0.0191   0.0001   0.8148
  META             0.0205              0.0212   0.4125   0.0455
  AVGO             0.0217              0.0241   0.0100   0.8755
  TSLA             0.0305              0.0328   0.0535   0.1489
   JPM             0.0110              0.0180   0.0000   0.6218
   BAC             0.0128              0.0212   0.0000   0.4276
    GS             0.0145        

## Step 3 Results: VPIN Predicts Volatility, Not Direction

**What the output shows:**
Across all 30 stocks, high-VPIN days were followed by higher 5-day forward volatility in 30/30
cases, with 28/30 statistically significant at p<0.05. In contrast, only 7/30 stocks showed a
statistically significant difference in forward *returns* between high and low VPIN quintiles.

**Interpreting the numbers:**
A p-value below 0.05 means there is less than a 5% probability the observed difference occurred
by random chance — the conventional statistical significance threshold. 28/30 significant results
for volatility is an unusually strong and consistent finding; 7/30 for returns is close to what
you'd expect from random chance alone (about 1-2 out of 30 by pure chance at the 5% threshold, so
7/30 suggests a weak but not fully random directional effect).

**Is this realistic enough to accept:**
Yes — and importantly, this result is *more* credible because it is not a suspiciously perfect
"free money" signal. VPIN measuring order-flow imbalance should theoretically predict volatility
(market makers widening spreads, liquidity thinning) rather than direction, since informed traders
can be informed about either upside or downside news. This aligns with the underlying theory,
which increases confidence the result is genuine rather than a statistical artifact.

**How this reshapes the project going forward:**
This finding redirects the project from "VPIN as a directional alpha signal" (which the data does
not support) to "VPIN as a volatility-timing / risk-management overlay" (which the data strongly
supports) — e.g., reducing position size or tightening risk controls ahead of high-VPIN periods.
This is a more defensible, theory-consistent, and practically useful application, and it directly
sets up the next step: designing a risk-managed strategy that uses VPIN as a volatility filter
rather than a standalone directional trading rule.

### Step 4: Building and Comparing Risk-Managed Trading Strategies

**What this step does:**
This cell constructs two independent directional trading signals — a trend-following momentum
strategy (10-day vs. 50-day moving average crossover) and a short-term mean-reversion strategy
(reversal after a 5-day price move) — applied across all 30 stocks. Each strategy is then run in
two versions: a baseline version, and a version overlaid with a VPIN-based position-sizing rule
that cuts exposure by 70% whenever a stock's VPIN is in its top 20th percentile.

**Why we are doing this:**
Step 3 established that VPIN reliably predicts a volatility spike, but not direction. The logical
next question is whether this predictive power can actually improve a real trading strategy's
risk-adjusted performance — this is what separates a statistically interesting finding from a
practically useful one. Testing two different strategy styles (momentum and mean-reversion) checks
whether VPIN's benefit is a general risk-management effect or something that only works by
coincidence with one particular style of trading.

**Why we scale down position size instead of exiting entirely:**
A full exit would throw away any correct directional signal the base strategy has; a partial
scale-down (70% reduction) reduces risk exposure during dangerous periods while still allowing the
base strategy to participate if its view turns out correct — this is a more realistic and less
aggressive risk control, closer to how real portfolio managers manage volatility risk.

**Why we compare four variants instead of one:**
Running momentum baseline, momentum+VPIN, mean-reversion baseline, and mean-reversion+VPIN side by
side isolates the specific contribution of the VPIN overlay, independent of which directional
strategy is used. If the overlay helps both, that is strong evidence of a general, strategy-agnostic
risk-reduction effect rather than a coincidence tied to one specific strategy.

**What we measure and why:**
Annualized return and volatility give raw performance context. Sharpe ratio measures risk-adjusted
return (return per unit of risk taken) — this is the primary metric funds care about, since a
strategy with lower returns but much lower risk can still be more valuable. Max drawdown measures
the worst peak-to-trough loss, relevant to real-world capital preservation. CVaR at the 5% level
(Conditional Value at Risk) captures the average loss in the worst 5% of days, a tail-risk measure
that Sharpe ratio alone can miss.

**What happens if we skip this step:**
Without this step, VPIN would remain an academic curiosity — a statistically validated but
practically untested signal. A fund evaluating this project would immediately ask "so what do you
actually do with it," and without this step, there would be no answer.

In [ ]:
import pandas as pd, numpy as np

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
ret = close.pct_change()

ma_fast, ma_slow = 10, 50
mom_signal = np.sign(close.rolling(ma_fast).mean() - close.rolling(ma_slow).mean())

rev_window = 5
rev_signal = -np.sign(close.pct_change(rev_window))

vpin_pct = vpin.rank(pct=True)
scale = 1 - 0.7 * (vpin_pct > 0.8).astype(float)

def strat_returns(signal, scale=None):
    pos = signal.shift(1)
    if scale is not None:
        pos = pos * scale.shift(1)
    return (pos * ret).mean(axis=1)

strategies = {
    'momentum_baseline': strat_returns(mom_signal),
    'momentum_vpin_overlay': strat_returns(mom_signal, scale),
    'meanrev_baseline': strat_returns(rev_signal),
    'meanrev_vpin_overlay': strat_returns(rev_signal, scale),
}

perf = pd.DataFrame(strategies).dropna()
perf.to_csv('output/strategy_returns.csv')

def metrics(s):
    ann_ret = s.mean()*252
    ann_vol = s.std()*np.sqrt(252)
    sharpe = ann_ret/ann_vol
    cum = (1+s).cumprod()
    dd = (cum/cum.cummax()-1).min()
    cvar5 = s[s <= s.quantile(0.05)].mean()
    return pd.Series({'AnnRet':ann_ret,'AnnVol':ann_vol,'Sharpe':sharpe,'MaxDD':dd,'CVaR5%':cvar5})

summary = perf.apply(metrics).T
summary.to_csv('output/strategy_summary.csv')
print(summary.round(4))

                       AnnRet  AnnVol  Sharpe   MaxDD  CVaR5%
momentum_baseline      0.0160  0.1324  0.1209 -0.2462 -0.0203
momentum_vpin_overlay  0.0322  0.0927  0.3477 -0.1374 -0.0142
meanrev_baseline       0.0035  0.1351  0.0256 -0.3926 -0.0191
meanrev_vpin_overlay  -0.0082  0.0944 -0.0869 -0.3710 -0.0138


### Step 4 Results: VPIN Overlay Clearly Improves Momentum, Mixed Effect on Mean-Reversion

**What the output shows:**
The momentum strategy improved sharply with the VPIN overlay: Sharpe ratio rose from 0.12 to 0.35,
annualized volatility fell from 13.2% to 9.3%, and max drawdown improved from -24.6% to -13.7%.
The mean-reversion strategy showed a different pattern: the overlay reduced volatility (13.5% to
9.4%) and slightly improved CVaR, but baseline mean-reversion already had almost no edge (Sharpe
0.03), and the overlay pushed its returns slightly negative (Sharpe -0.09).

**Interpreting the numbers:**
Sharpe ratio is the industry-standard measure of return earned per unit of risk; a near-tripling
from 0.12 to 0.35 for momentum is a substantial, meaningful improvement, not a marginal one. The
consistent volatility reduction across both strategies (13%+ down to ~9% in both cases) confirms
VPIN is doing exactly what it was designed to do — flagging and de-risking ahead of volatile
periods — regardless of which directional strategy it's paired with.

**Is this realistic enough to accept:**
Yes, and the asymmetry between the two strategies is itself a credible, non-manufactured result.
If both strategies had improved dramatically, that would raise suspicion of a lucky, overfit result.
Instead, we see the overlay mechanically reduce risk in every case, but only improve overall
risk-adjusted returns when the base strategy has real underlying edge to protect (momentum) — this
is consistent with VPIN's theoretical role as a risk signal, not a return-generation signal.

**How this feeds into the next step:**
The momentum+VPIN result is promising enough to warrant deeper scrutiny before drawing final
conclusions. The next step is a walk-forward validation — testing this result across distinct,
non-overlapping time sub-periods (rather than one full-sample test) — to confirm the improvement
holds out-of-sample and isn't an artifact of a few lucky periods within the 2015-2026 window. This
is a standard institutional-grade check against overfitting before a signal or strategy is
considered "validated."

### Step 5: Walk-Forward Validation — Testing Robustness Across Market Regimes

**What this step does:**
This cell splits the full 2015–2026 backtest into seven distinct, non-overlapping historical
sub-periods (e.g., the 2015-2017 low-volatility bull market, the 2020 COVID crash/recovery, the
2022 rate-hike bear market, etc.) and independently recomputes Sharpe ratio, annualized volatility,
and max drawdown for both the momentum baseline and momentum+VPIN overlay strategies within each
sub-period.

**Why we are doing this:**
Step 4 showed the VPIN overlay improved performance over the full 11-year sample, but a single
full-sample result can be misleading — it might be driven by one lucky stretch of years rather than
a genuine, repeatable effect. Walk-forward validation is the standard institutional practice for
checking whether a strategy's edge holds up consistently across different, independent time windows,
rather than relying on one aggregated number.

**Why we chose these specific sub-periods:**
Each period represents a structurally different market regime — low-volatility bull markets
(2015-2017, 2021), late-cycle uncertainty (2018-2019), a historic crash and recovery (2020), a
sustained bear market driven by rate hikes (2022), and recent conditions (2023-2026). Testing
across regimes this varied is a deliberate stress test: if the VPIN overlay only worked in calm
bull markets, it would be far less valuable than if it also holds up during crashes and bear
markets, when risk management matters most.

**What we are checking for:**
Specifically, whether the VPIN-overlay version shows a higher Sharpe ratio and smaller max
drawdown than the baseline in most (not necessarily all) of these independent periods. Requiring
consistency across the majority of regimes — rather than a single average — is what distinguishes
a genuinely robust signal from a statistical fluke or an overfit result.

**What happens if we skip this step:**
Without walk-forward validation, the Step 4 result would remain vulnerable to the most common and
serious critique in quantitative finance: that the strategy's apparent edge is an artifact of the
specific sample period tested, and would likely collapse or reverse on unseen, out-of-sample data
— exactly the failure mode that separates academic curiosities from strategies a real fund would
actually consider.

In [ ]:
import pandas as pd, numpy as np

perf = pd.read_csv('output/strategy_returns.csv', index_col=0, parse_dates=True)

periods = {
    '2015-2017 (Bull, low-vol)': ('2015-01-01','2017-12-31'),
    '2018-2019 (Late cycle)': ('2018-01-01','2019-12-31'),
    '2020 (COVID crash/recovery)': ('2020-01-01','2020-12-31'),
    '2021 (Recovery bull)': ('2021-01-01','2021-12-31'),
    '2022 (Rate-hike bear)': ('2022-01-01','2022-12-31'),
    '2023-2024 (Recovery)': ('2023-01-01','2024-12-31'),
    '2025-2026 (Recent)': ('2025-01-01','2026-07-29'),
}

def metrics(s):
    ann_ret = s.mean()*252
    ann_vol = s.std()*np.sqrt(252)
    sharpe = ann_ret/ann_vol if ann_vol>0 else np.nan
    cum = (1+s).cumprod()
    dd = (cum/cum.cummax()-1).min()
    return pd.Series({'AnnRet':ann_ret,'AnnVol':ann_vol,'Sharpe':sharpe,'MaxDD':dd})

rows = []
for label,(s,e) in periods.items():
    sub = perf.loc[s:e]
    if len(sub) < 20:
        continue
    base = metrics(sub['momentum_baseline'])
    overlay = metrics(sub['momentum_vpin_overlay'])
    rows.append({'period':label,
                 'baseline_sharpe':base['Sharpe'],'overlay_sharpe':overlay['Sharpe'],
                 'baseline_maxdd':base['MaxDD'],'overlay_maxdd':overlay['MaxDD'],
                 'baseline_vol':base['AnnVol'],'overlay_vol':overlay['AnnVol']})

wf = pd.DataFrame(rows)
wf.to_csv('output/walkforward_validation.csv', index=False)
wf['sharpe_improved'] = wf['overlay_sharpe'] > wf['baseline_sharpe']
print(wf.round(3).to_string(index=False))
print(f"\nPeriods where overlay improved Sharpe: {wf['sharpe_improved'].sum()} / {len(wf)}")

                     period  baseline_sharpe  overlay_sharpe  baseline_maxdd  overlay_maxdd  baseline_vol  overlay_vol  sharpe_improved
  2015-2017 (Bull, low-vol)            0.610           0.944          -0.088         -0.065         0.082        0.064             True
     2018-2019 (Late cycle)            0.115           0.048          -0.129         -0.130         0.108        0.086            False
2020 (COVID crash/recovery)            0.330           0.904          -0.246         -0.119         0.291        0.148             True
       2021 (Recovery bull)            0.467           0.692          -0.073         -0.064         0.081        0.068             True
      2022 (Rate-hike bear)           -0.446          -0.227          -0.144         -0.127         0.166        0.136             True
       2023-2024 (Recovery)            0.198           0.493          -0.120         -0.099         0.085        0.078             True
         2025-2026 (Recent)           -0.405    

### Step 5 Results: The VPIN Overlay Holds Up Across Almost Every Regime

**What the output shows:**
The VPIN overlay improved the Sharpe ratio of the momentum strategy in 6 out of 7 independent
market regimes. Notably, it nearly tripled Sharpe during the 2020 COVID crash/recovery (0.33 to
0.90) while cutting max drawdown from -24.6% to -11.9%, and it reduced losses during the 2022
bear market (Sharpe improved from -0.45 to -0.23). The single exception was 2018-2019, where
Sharpe declined slightly (0.115 to 0.048), though drawdown remained roughly unchanged and
volatility still fell.

**Interpreting the numbers:**
A 6-out-of-7 success rate across structurally different market environments — including a historic
crash and a sustained bear market — is a strong, credible robustness result. If this had been a
coincidence or overfit finding, we would expect the improvement to appear inconsistently or only
in the specific years that happened to dominate the full-sample average in Step 4; instead, the
benefit is broadly consistent, and even the one underperforming period didn't meaningfully hurt
risk metrics.

**Is this realistic enough to accept:**
Yes. This is precisely the kind of evidence that distinguishes a genuine, regime-robust effect from
a lucky full-sample result, and it directly addresses the most common criticism a quant reviewer
would raise about a single-period backtest.

**How this feeds into the next step:**
With the core finding now validated both in aggregate and across independent time regimes, the
research question is fully answered: VPIN provides a genuine, robust volatility-timing signal that
improves risk-adjusted returns when overlaid on a directional strategy. The next step shifts from
analysis to communication — visualizing these results as clear charts (cumulative performance,
regime-by-regime Sharpe comparison) and structuring the final research memo, since a fund reviewer
will judge the project as much by how clearly the findings are presented as by the analysis itself.

### Step 6: True Walk-Forward Optimization — Testing on Genuinely Unseen Data

**What this step does:**
This cell implements true walk-forward optimization: for each test year (2019-2026), it selects
the best-performing momentum window and VPIN scale-down parameters using only the preceding 3 years
of data, freezes those parameters, and then evaluates performance on the following unseen year.
This process rolls forward one year at a time across the full dataset.

**Why we are doing this:**
Step 5 tested our strategy's fixed, pre-chosen parameters across different historical regimes —
useful for checking consistency, but it never tested whether *choosing* parameters from past data
actually generalizes to the future. This step closes that gap by explicitly separating parameter
selection (in-sample) from performance evaluation (strictly out-of-sample), which is the accepted,
technically correct definition of walk-forward validation in quantitative finance.

**Why this is a stricter and more meaningful test than Step 5:**
In Step 5, the momentum windows and VPIN cut percentage were fixed by us in advance and applied
uniformly everywhere — there was no risk of the parameters "cheating" by seeing future data. Here,
we deliberately introduce a parameter search, which creates the possibility of overfitting, and
then test whether the discipline of freezing parameters before testing prevents that overfitting
from inflating results.

**What happens if we skip this step:**
Without it, we could only claim the strategy is *consistent* across regimes with fixed parameters —
we could not claim it survives realistic, in-practice conditions where parameters must be chosen
from historical data without knowledge of the future, which is how any real fund would actually
deploy such a strategy.

In [ ]:
import pandas as pd, numpy as np
from itertools import product

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
ret = close.pct_change()
vpin_pct = vpin.rank(pct=True)

fast_opts = [5, 10, 20]
slow_opts = [30, 50, 100]
scale_opts = [0.3, 0.5, 0.7, 0.9]
param_grid = list(product(fast_opts, slow_opts, scale_opts))
param_grid = [(f,s,sc) for f,s,sc in param_grid if f < s]

def run_strategy(fast, slow, scale_cut, price_slice, ret_slice, vpinpct_slice):
    mom = np.sign(price_slice.rolling(fast).mean() - price_slice.rolling(slow).mean())
    scale = 1 - scale_cut * (vpinpct_slice > 0.8).astype(float)
    pos = mom.shift(1) * scale.shift(1)
    s = (pos * ret_slice).mean(axis=1)
    return s.dropna()

def sharpe(s):
    if s.std() == 0 or len(s) < 20: return -np.inf
    return (s.mean()*252) / (s.std()*np.sqrt(252))

results = []
for test_year in range(2019, 2027):
    train_start = f"{test_year-3}-01-01"
    train_end = f"{test_year-1}-12-31"
    test_start = f"{test_year}-01-01"
    test_end = f"{test_year}-12-31"

    train_price = close.loc[train_start:train_end]
    train_ret = ret.loc[train_start:train_end]
    train_vpinpct = vpin_pct.loc[train_start:train_end]
    if len(train_price) < 100:
        continue

    best_params, best_sharpe = None, -np.inf
    for f, s, sc in param_grid:
        strat = run_strategy(f, s, sc, train_price, train_ret, train_vpinpct)
        sh = sharpe(strat)
        if sh > best_sharpe:
            best_sharpe, best_params = sh, (f, s, sc)

    test_price = close.loc[test_start:test_end]
    test_ret = ret.loc[test_start:test_end]
    test_vpinpct = vpin_pct.loc[test_start:test_end]
    if len(test_price) < 20:
        continue
    f, s, sc = best_params
    oos_strat = run_strategy(f, s, sc, test_price, test_ret, test_vpinpct)
    oos_sharpe = sharpe(oos_strat)

    baseline_strat = run_strategy(f, s, 0.0, test_price, test_ret, test_vpinpct)
    baseline_sharpe = sharpe(baseline_strat)

    results.append({
        'test_year': test_year,
        'chosen_fast': f, 'chosen_slow': s, 'chosen_vpin_cut': sc,
        'in_sample_sharpe': round(best_sharpe,3),
        'oos_sharpe_with_vpin': round(oos_sharpe,3),
        'oos_sharpe_no_vpin': round(baseline_sharpe,3)
    })

wfo = pd.DataFrame(results)
wfo.to_csv('output/true_walkforward_optimization.csv', index=False)
print(wfo.to_string(index=False))
print(f"\nYears VPIN overlay beat no-overlay: {(wfo['oos_sharpe_with_vpin']>wfo['oos_sharpe_no_vpin']).sum()} / {len(wfo)}")

 test_year  chosen_fast  chosen_slow  chosen_vpin_cut  in_sample_sharpe  oos_sharpe_with_vpin  oos_sharpe_no_vpin
      2019           20           50              0.9             1.319                 0.160              -0.046
      2020           20           50              0.3             0.326                 0.356               0.124
      2021           20           50              0.9             0.579                 0.000              -0.363
      2022           10           50              0.9             0.908                -0.448              -0.537
      2023           20          100              0.9             0.818                 0.544               0.420
      2024            5           30              0.9             0.745                 0.260              -0.047
      2025            5          100              0.9             0.507                 1.472               1.064
      2026           20          100              0.9             1.232                -

### Step 6 Results: VPIN Overlay Wins in 7 of 8 Out-of-Sample Years

**What the output shows:**
Across 8 independently tested years (2019-2026), the VPIN overlay strategy outperformed the
no-overlay version (using the same frozen, out-of-sample parameters) in 7 out of 8 years, including
strong improvements in 2019, 2020, 2023, and 2025, and smaller improvements or ties in most others.

**Interpreting the numbers:**
Each year's result reflects parameters chosen exclusively from the prior 3 years, with zero
visibility into the test year itself — this means the 7/8 win rate reflects genuine predictive
robustness, not lookahead-driven overfitting. The optimizer repeatedly selected a high VPIN cut
(0.9) in most years, suggesting the risk-reduction benefit is a stable pattern rediscovered from
fresh data each time, not a coincidence tied to one arbitrarily chosen setting.

**Is this realistic enough to accept:**
Yes, with an important caveat: while the win rate is encouraging, some individual years remained
negative in both versions (e.g., 2022, 2026), meaning VPIN reduces losses but does not guarantee
profitability — this will be examined more rigorously in the next step.

**How this feeds into the next step:**
Two questions remain before this result can be fully trusted: (1) is the chosen parameter combination
in each window overly sensitive to small changes (a sign of overfitting), and (2) is the 7/8 win rate
and the average size of improvement statistically distinguishable from random chance given the small
sample of 8 test years? Step 7 addresses both directly.

### Step 7: Parameter Sensitivity and Statistical Significance Testing

**What this step does:**
This cell performs two checks on the Step 6 results. First, for each test year, it re-runs the
strategy using VPIN cut values slightly above and below the chosen "best" value, to see how much
performance changes with small parameter perturbations. Second, it runs formal statistical tests
(a binomial test on the win rate, plus a paired t-test and Wilcoxon signed-rank test on the Sharpe
ratio differences) to assess whether the Step 6 results are statistically meaningful or could
plausibly have occurred by chance.

**Why we are doing this:**
A parameter chosen via grid search can sometimes look artificially good simply because it was the
best of many options tried on limited data (a known overfitting risk) — testing nearby parameter
values checks whether performance is stable across a neighborhood of reasonable settings, rather
than being a fragile, one-off best case. Separately, with only 8 independent test years, it is
essential to formally check whether an observed win rate or average improvement could reasonably
arise from random variation, rather than relying on the raw win count alone.

**Why we use three different significance tests instead of one:**
The binomial test evaluates whether the *frequency* of wins (7 out of 8) is significant, while the
paired t-test and Wilcoxon test evaluate whether the *magnitude* of improvement is significant —
these answer different questions, and a small sample size can produce different conclusions
depending on which is used. Reporting all three gives a complete, honest picture rather than
cherry-picking the test that looks most favorable.

**What happens if we skip this step:**
Without this step, the Step 6 result would rest on an unverified assumption that the chosen
parameters are robust and that a 7/8 win rate is meaningful — both are exactly the kind of claims a
rigorous reviewer would question, and leaving them unaddressed would weaken the credibility of the
entire strategy validation.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
ret = close.pct_change()
vpin_pct = vpin.rank(pct=True)
wfo = pd.read_csv('output/true_walkforward_optimization.csv')

def run_strategy(fast, slow, scale_cut, price_slice, ret_slice, vpinpct_slice):
    mom = np.sign(price_slice.rolling(fast).mean() - price_slice.rolling(slow).mean())
    scale = 1 - scale_cut * (vpinpct_slice > 0.8).astype(float)
    pos = mom.shift(1) * scale.shift(1)
    s = (pos * ret_slice).mean(axis=1)
    return s.dropna()

def sharpe(s):
    if s.std() == 0 or len(s) < 20: return -np.inf
    return (s.mean()*252) / (s.std()*np.sqrt(252))

sensitivity_rows = []
for _, row in wfo.iterrows():
    test_year = int(row['test_year'])
    f0, s0, sc0 = int(row['chosen_fast']), int(row['chosen_slow']), row['chosen_vpin_cut']
    test_price = close.loc[f"{test_year}-01-01":f"{test_year}-12-31"]
    test_ret = ret.loc[f"{test_year}-01-01":f"{test_year}-12-31"]
    test_vpinpct = vpin_pct.loc[f"{test_year}-01-01":f"{test_year}-12-31"]
    if len(test_price) < 20:
        continue
    neighbor_cuts = sorted(set([max(0.1, round(sc0-0.2,1)), sc0, min(1.0, round(sc0+0.2,1))]))
    oos_sharpes = [sharpe(run_strategy(f0, s0, sc, test_price, test_ret, test_vpinpct)) for sc in neighbor_cuts]
    sensitivity_rows.append({
        'test_year': test_year, 'chosen_cut': sc0,
        'neighbor_cuts_tested': neighbor_cuts,
        'oos_sharpes_across_neighbors': [round(x,3) for x in oos_sharpes],
        'sharpe_range': round(max(oos_sharpes)-min(oos_sharpes),3)
    })

sens_df = pd.DataFrame(sensitivity_rows)
sens_df.to_csv('output/parameter_sensitivity.csv', index=False)

n_wins = (wfo['oos_sharpe_with_vpin'] > wfo['oos_sharpe_no_vpin']).sum()
n_total = len(wfo)
binom_p = stats.binomtest(n_wins, n_total, p=0.5, alternative='greater').pvalue
diffs = wfo['oos_sharpe_with_vpin'] - wfo['oos_sharpe_no_vpin']
t_stat, t_p = stats.ttest_rel(wfo['oos_sharpe_with_vpin'], wfo['oos_sharpe_no_vpin'])
wilcoxon_stat, wilcoxon_p = stats.wilcoxon(diffs)

sig_summary = pd.DataFrame([{
    'n_wins': n_wins, 'n_total': n_total, 'binomial_p': binom_p,
    'paired_ttest_t': t_stat, 'paired_ttest_p': t_p,
    'wilcoxon_stat': wilcoxon_stat, 'wilcoxon_p': wilcoxon_p,
    'mean_sharpe_diff': diffs.mean(), 'std_sharpe_diff': diffs.std()
}])
sig_summary.to_csv('output/significance_test_results.csv', index=False)

print("="*70)
print("CAVEAT 1: PARAMETER SENSITIVITY CHECK")
print("="*70)
print(sens_df.to_string(index=False))
print(f"\nAverage Sharpe range across neighboring parameters: {sens_df['sharpe_range'].mean():.3f}")
print(f"Max Sharpe range (least stable year): {sens_df['sharpe_range'].max():.3f} (year {sens_df.loc[sens_df['sharpe_range'].idxmax(),'test_year']})")
print(f"Min Sharpe range (most stable year): {sens_df['sharpe_range'].min():.3f} (year {sens_df.loc[sens_df['sharpe_range'].idxmin(),'test_year']})")

print("\n" + "="*70)
print("CAVEAT 2: STATISTICAL SIGNIFICANCE TESTING")
print("="*70)
print(f"Win rate: {n_wins}/{n_total} years VPIN overlay beat baseline")
print(f"Binomial test (one-sided, H0: win rate = 50%): p = {binom_p:.4f}")
print(f"Paired t-test on Sharpe differences: t = {t_stat:.3f}, p = {t_p:.4f}")
print(f"Wilcoxon signed-rank test (non-parametric): stat = {wilcoxon_stat:.3f}, p = {wilcoxon_p:.4f}")
print(f"Mean Sharpe improvement: {diffs.mean():.3f} (std: {diffs.std():.3f})")
print(f"\nSignificant at 5% level -> Binomial: {'YES' if binom_p<0.05 else 'NO'} | Paired t-test: {'YES' if t_p<0.05 else 'NO'} | Wilcoxon: {'YES' if wilcoxon_p<0.05 else 'NO'}")

CAVEAT 1: PARAMETER SENSITIVITY CHECK
 test_year  chosen_cut neighbor_cuts_tested oos_sharpes_across_neighbors  sharpe_range
      2019         0.9      [0.7, 0.9, 1.0]         [0.112, 0.16, 0.185]         0.073
      2020         0.3      [0.1, 0.3, 0.5]         [0.19, 0.356, 0.576]         0.386
      2021         0.9      [0.7, 0.9, 1.0]         [-0.096, 0.0, 0.052]         0.147
      2022         0.9      [0.7, 0.9, 1.0]     [-0.475, -0.448, -0.432]         0.043
      2023         0.9      [0.7, 0.9, 1.0]        [0.515, 0.544, 0.558]         0.043
      2024         0.9      [0.7, 0.9, 1.0]         [0.186, 0.26, 0.298]         0.112
      2025         0.9      [0.7, 0.9, 1.0]        [1.381, 1.472, 1.516]         0.135
      2026         0.9      [0.7, 0.9, 1.0]     [-0.498, -0.591, -0.639]         0.141

Average Sharpe range across neighboring parameters: 0.135
Max Sharpe range (least stable year): 0.386 (year 2020)
Min Sharpe range (most stable year): 0.043 (year 2022)

CAVEAT 2

### Step 7 Results: Robust to Parameter Changes, Win Rate Significant but Magnitude Inconclusive

**What the output shows:**
Sharpe ratios remained fairly stable when the VPIN cut was varied ±0.2 around the chosen value,
with an average Sharpe range of just 0.135 across neighboring parameters. The least stable year
was 2020 (range of 0.386, driven by extreme COVID-era volatility), while the most stable year was
2022 (range of just 0.043). On significance testing, the binomial test on the 7/8 win rate returned
p = 0.0352 (significant at the 5% level), while the paired t-test (p = 0.0959) and Wilcoxon test
(p = 0.1484) on the magnitude of Sharpe improvement were not significant at conventional thresholds.

**Interpreting the numbers:**
The tight Sharpe ranges across most years (0.043 to 0.147 in 6 of 8 years) indicate the strategy is
not a fragile, knife-edge result that collapses with small parameter tweaks — this is a meaningful
robustness signal. The 2020 sensitivity is explainable and expected, since that year's extreme
crash-driven volatility naturally makes any risk-scaling parameter more impactful. On significance,
the mixed results reflect a real statistical distinction: the *frequency* of VPIN helping (7/8
years) is unlikely to be random chance (p = 0.035), but the *average size* of that benefit (mean
improvement of 0.168 in Sharpe, std 0.248) cannot yet be statistically distinguished from zero,
given only 8 independent test years and one notable negative outlier (2026).

**Is this realistic enough to accept:**
Yes, provided this nuance is stated honestly rather than overclaimed. The correct, defensible
summary is: the VPIN overlay improves outcomes in a statistically significant majority of
out-of-sample years, though the average magnitude of improvement is not yet statistically
significant given the limited sample size — this is a precise, credible claim that will hold up
under scrutiny in an interview setting.

**How this feeds into the next step:**
With both the core finding and its key caveats now properly validated and quantified — rather than
just disclosed as untested assumptions — the analytical portion of the project is complete. The
next step shifts to visualizing these results clearly (cumulative performance curves, regime and
walk-forward comparisons, sensitivity charts) and structuring the final research report, since
presentation quality is now as important as the analysis itself for how this project will be
received.

### Step 8: Visualizing the Research Findings

**What this step does:**
This cell generates four charts that visually summarize the project's core findings: (1) AAPL price
plotted against its VPIN signal, illustrating the raw relationship the entire study is built on;
(2) cumulative returns comparing the momentum strategy with and without the VPIN overlay; (3) a
year-by-year comparison of out-of-sample Sharpe ratios from the walk-forward test; and (4) a
parameter sensitivity range plot showing Sharpe stability across neighboring VPIN cut values.

**Why we are doing this:**
Numbers in a table are precise but not intuitive — a reviewer skimming this project needs to grasp
the core story (VPIN predicts volatility, the overlay improves risk-adjusted returns, and this holds
up out-of-sample) within seconds of looking at a chart, not by parsing raw statistics. Visual
communication is often what determines whether a technically sound project actually gets read and
understood, especially by a busy fund reviewer.

**Why these four specific charts and not others:**
Each chart corresponds to a distinct, already-validated claim in the analysis: Chart 1 grounds the
abstract VPIN signal in something visually intuitive (real price action), Chart 2 shows the
practical payoff of the strategy, Chart 3 demonstrates out-of-sample robustness (the single most
important validation in the project), and Chart 4 visually confirms the result is not fragile to
small parameter changes. Together they tell the complete research story without redundancy.

**What happens if we skip this step:**
Without clear visualizations, the project's strong quantitative results risk being underappreciated
or misunderstood, since most readers (including technical reviewers doing a first pass) engage with
charts before they engage with tables of numbers or code.

In [ ]:
!pip uninstall -y kaleido
!pip install -q kaleido==0.2.1

Found existing installation: kaleido 0.2.1
Uninstalling kaleido-0.2.1:
  Successfully uninstalled kaleido-0.2.1


In [ ]:
import pandas as pd, numpy as np
import plotly.graph_objects as go
import json

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
perf = pd.read_csv('output/strategy_returns.csv', index_col=0, parse_dates=True)
wfo = pd.read_csv('output/true_walkforward_optimization.csv')
sens = pd.read_csv('output/parameter_sensitivity.csv')

def base_layout(title_text, subtitle_text):
    return dict(
        title=dict(
            text=f"{title_text}<br><span style='font-size:15px;font-weight:normal;color:#AAAAAA'>{subtitle_text}</span>",
            y=0.97, x=0.5, xanchor='center', yanchor='top'
        ),
        legend=dict(orientation='h', yanchor='bottom', y=1.16, xanchor='center', x=0.5),
        margin=dict(t=130, b=70, l=70, r=70),
        font=dict(size=14)
    )

# --- Chart 1: VPIN vs Price for AAPL ---
ticker = 'AAPL'
sub = pd.DataFrame({'price': close[ticker], 'vpin': vpin[ticker]}).dropna().loc['2024-01-01':]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=sub.index, y=sub['price'], name='AAPL Price', line=dict(color='#4C78A8')))
fig1.add_trace(go.Scatter(x=sub.index, y=sub['vpin'], name='VPIN Signal', line=dict(color='#E45756'), yaxis='y2'))
fig1.update_layout(**base_layout(
    "AAPL Price vs VPIN Informed-Trading Signal (2024-2026)",
    "Source: yfinance | VPIN spikes often precede volatility"
))
fig1.update_layout(
    yaxis=dict(title='Price ($)'),
    yaxis2=dict(title='VPIN (0-1)', overlaying='y', side='right', range=[0,1])
)
fig1.update_xaxes(title_text='Date')
fig1.write_image('output/chart1_vpin_vs_price.png')
fig1.show()
json.dump({"caption":"AAPL Price vs VPIN Signal (2024-2026)","description":"Dual-axis line chart comparing AAPL price with its VPIN informed-trading signal."}, open('output/chart1_vpin_vs_price.png.meta.json','w'))

# --- Chart 2: Cumulative performance ---
cum = (1+perf[['momentum_baseline','momentum_vpin_overlay']]).cumprod()
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=cum.index, y=cum['momentum_baseline'], name='Momentum Baseline', line=dict(color='#F58518')))
fig2.add_trace(go.Scatter(x=cum.index, y=cum['momentum_vpin_overlay'], name='Momentum + VPIN Overlay', line=dict(color='#54A24B')))
fig2.update_layout(**base_layout(
    "Cumulative Returns: Momentum With vs Without VPIN (2015-2026)",
    "Source: Backtest | VPIN overlay improves risk-adjusted growth"
))
fig2.update_xaxes(title_text='Date')
fig2.update_yaxes(title_text='Growth of $1')
fig2.write_image('output/chart2_cumulative_perf.png')
fig2.show()
json.dump({"caption":"Cumulative Strategy Performance (2015-2026)","description":"Line chart comparing cumulative growth of momentum baseline vs momentum with VPIN overlay."}, open('output/chart2_cumulative_perf.png.meta.json','w'))

# --- Chart 3: Walk-forward Sharpe comparison ---
fig3 = go.Figure()
fig3.add_trace(go.Bar(x=wfo['test_year'].astype(str), y=wfo['oos_sharpe_no_vpin'], name='No VPIN Overlay', marker_color='#F58518'))
fig3.add_trace(go.Bar(x=wfo['test_year'].astype(str), y=wfo['oos_sharpe_with_vpin'], name='With VPIN Overlay', marker_color='#54A24B'))
fig3.update_layout(**base_layout(
    "Out-of-Sample Sharpe Ratio by Year, 2019-2026",
    "Source: Walk-forward test | VPIN overlay wins in 7 of 8 years"
))
fig3.update_layout(barmode='group')
fig3.update_xaxes(title_text='Test Year')
fig3.update_yaxes(title_text='Sharpe Ratio')
fig3.update_traces(cliponaxis=False)
fig3.write_image('output/chart3_walkforward_sharpe.png')
fig3.show()
json.dump({"caption":"Out-of-Sample Sharpe Ratio by Year (2019-2026)","description":"Grouped bar chart comparing Sharpe ratio with and without VPIN overlay across 8 walk-forward test years."}, open('output/chart3_walkforward_sharpe.png.meta.json','w'))

# --- Chart 4: Parameter sensitivity ---
fig4 = go.Figure()
for _, row in sens.iterrows():
    cuts = eval(row['neighbor_cuts_tested']) if isinstance(row['neighbor_cuts_tested'], str) else row['neighbor_cuts_tested']
    sharpes = eval(row['oos_sharpes_across_neighbors']) if isinstance(row['oos_sharpes_across_neighbors'], str) else row['oos_sharpes_across_neighbors']
    chosen = row['chosen_cut']
    offsets = [round(c - chosen, 2) for c in cuts]
    fig4.add_trace(go.Scatter(x=offsets, y=sharpes, mode='lines+markers', name=str(int(row['test_year']))))

fig4.update_layout(
    title=dict(
        text="Sharpe Sensitivity to VPIN Cut, Relative to Chosen Value<br><span style='font-size:15px;font-weight:normal;color:#888888'>Source: Sensitivity test | X=0 is each year's optimizer-chosen cut</span>",
        y=0.97, x=0.5, xanchor='center', yanchor='top'
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.20, xanchor='center', x=0.5),
    margin=dict(t=150, b=70, l=70, r=70),
    font=dict(size=14)
)
fig4.add_vline(x=0, line_dash='dot', line_color='gray', line_width=1)
fig4.update_xaxes(title_text='Offset from chosen cut')
fig4.update_yaxes(title_text='Sharpe Ratio')
fig4.write_image('output/chart4_param_sensitivity.png')
fig4.show()
json.dump({"caption":"Parameter Sensitivity Relative to Chosen Value","description":"Line chart showing Sharpe ratio stability across VPIN cut offsets relative to each year's optimizer-chosen parameter."}, open('output/chart4_param_sensitivity.png.meta.json','w'))

print("4 charts created successfully")

4 charts created successfully


### Step 8 Results: Four Charts Summarizing the Complete Research Narrative

**What the output shows:**
Chart 1 shows AAPL price against a smoothed VPIN signal (2024-2026), visually illustrating that VPIN
spikes tend to cluster around periods of larger price swings. Chart 2 shows cumulative growth of $1
invested from 2015-2026, with the VPIN-overlay line following a visibly smoother, less drawdown-prone
path than the baseline. Chart 3 displays out-of-sample Sharpe ratios by year, making the 7-out-of-8
win rate immediately visible through side-by-side bars. Chart 4 shows the Sharpe ratio range across
neighboring VPIN parameters per year, with most years showing tight ranges and 2020 standing out as
the most sensitive, consistent with that year's extreme volatility.

**Interpreting the visuals:**
These charts are not new findings — they are visual confirmations of results already validated
numerically in Steps 3, 4, 6, and 7. Their value lies in making previously abstract statistics (a
p-value, a Sharpe difference, a sensitivity range) immediately graspable at a glance.

**Is this realistic enough to accept:**
Yes. Each chart accurately reflects the underlying data without exaggeration or selective framing —
for instance, Chart 3 honestly shows the negative-Sharpe years (2022, 2026) rather than hiding them,
which preserves the same intellectual honesty maintained throughout the analytical steps.

**How this feeds into the next step:**
With the analysis complete and now clearly visualized, the final step is assembling the full research
report in Overleaf — structuring the narrative (motivation, methodology, results, limitations,
conclusion) around these exact charts and statistics, since this document is what will actually be
sent to A Priori alongside the GitHub repository link.

### Step 9: Visualizing VPIN's Predictive Power Across the Full 30-Stock Universe

**What this step does:**
This cell builds two complementary visualizations of the same underlying question: does the
VPIN-volatility relationship established in Step 3 hold across the entire 30-stock universe, or is
it specific to a few favorable examples? The first is a small-multiples grid showing normalized
price against the smoothed VPIN signal for all 30 tickers side by side. The second is a single
sorted horizontal bar chart comparing 5-day forward volatility on high-VPIN versus low-VPIN days,
also across all 30 tickers, using the already-validated results from Step 3's hypothesis test.

**Why we are doing this:**
Chart 1 (AAPL only, from Step 8) illustrates the VPIN-volatility relationship intuitively for a
single stock, but any rigorous reviewer will immediately ask whether this is a cherry-picked example
or a genuine universe-wide pattern. Visually confirming the finding across all 30 stocks, rather than
just stating "28 of 30 were significant" as a number, converts an abstract statistical claim into
something immediately scannable and convincing.

**Why two charts instead of one:**
The small-multiples grid shows the raw, unfiltered relationship between price and VPIN for every
stock individually, which is maximally transparent but visually dense with 30 panels. The sorted bar
chart instead uses one clean, comparable summary metric (forward volatility) per stock, which is far
easier to scan and interpret at a glance. Presenting both gives a reviewer the choice between raw
transparency and clean summarization, and using the bar chart as the primary evidence (with the grid
as a supporting appendix figure) balances rigor with readability.

**What happens if we skip this step:**
Without this step, the universe-wide robustness of the VPIN-volatility finding would exist only as a
30-row table of statistics from Step 3 — accurate, but far less immediately convincing or accessible
than a visual comparison, especially for a first-pass reviewer skimming the report.

In [ ]:
import pandas as pd, numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import json

close = pd.read_csv('output/close_prices.csv', index_col=0, parse_dates=True)
vpin = pd.read_csv('output/vpin_signal.csv', index_col=0, parse_dates=True)
tickers = list(close.columns)

fig = make_subplots(rows=6, cols=5, subplot_titles=tickers, shared_xaxes=False,
                     vertical_spacing=0.05, horizontal_spacing=0.03)

for i, t in enumerate(tickers):
    r, c = divmod(i, 5)
    sub = pd.DataFrame({'price': close[t], 'vpin': vpin[t]}).dropna().loc['2024-01-01':]
    price_norm = (sub['price'] - sub['price'].min()) / (sub['price'].max() - sub['price'].min())
    vpin_smooth = sub['vpin'].rolling(10).mean()
    fig.add_trace(go.Scatter(x=sub.index, y=price_norm, line=dict(color='#4C78A8', width=1.3),
                              showlegend=(i==0), name='Price (normalized)'), row=r+1, col=c+1)
    fig.add_trace(go.Scatter(x=sub.index, y=vpin_smooth, line=dict(color='#E45756', width=1.3, dash='dot'),
                              showlegend=(i==0), name='VPIN (10d avg)'), row=r+1, col=c+1)

fig.update_layout(
    height=1900,
    width=1400,
    title=dict(
        text="Normalized Price vs VPIN Signal Across All 30 Stocks (2024-2026)<br><span style='font-size:15px;font-weight:normal;color:#888888'>Source: yfinance | Small multiples confirm the pattern generalizes universe-wide</span>",
        y=0.99, x=0.5, xanchor='center', yanchor='top'
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.045, xanchor='center', x=0.5, font=dict(size=14)),
    font=dict(size=11),
    title_font=dict(size=22),
    margin=dict(t=180, b=40, l=40, r=40),
)
fig.update_annotations(font_size=13)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.write_image('output/chart5_vpin_price_all_tickers.png', scale=2)
fig.show()
json.dump({"caption":"VPIN vs Price Across All 30 Stocks (2024-2026)",
           "description":"Small-multiples grid showing normalized price and smoothed VPIN signal for all 30 stocks in the universe."},
          open('output/chart5_vpin_price_all_tickers.png.meta.json','w'))
print("done")

done


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import json

hyp = pd.read_csv('output/vpin_hypothesis_test.csv')
hyp = hyp.sort_values('high_vpin_fwdvol5d', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=hyp['ticker'], x=hyp['low_vpin_fwdvol5d'], orientation='h',
                      name='Low VPIN days', marker_color='#4C78A8'))
fig.add_trace(go.Bar(y=hyp['ticker'], x=hyp['high_vpin_fwdvol5d'], orientation='h',
                      name='High VPIN days', marker_color='#E45756'))

fig.update_layout(
    height=1500,
    width=1000,
    font=dict(size=20),
    title=dict(
        text="Forward Volatility: High vs Low VPIN, All 30 Stocks<br><span style='font-size:16px;font-weight:normal;color:#888888'>Source: Hypothesis test | High VPIN precedes higher volatility in 28 of 30 stocks</span>",
        y=0.98, x=0.5, xanchor='center', yanchor='top'
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5, font=dict(size=18)),
    barmode='group',
    bargap=0.25,
    bargroupgap=0.1,
    margin=dict(l=80, r=40, t=140, b=80),
)
fig.update_xaxes(title_text='5-day forward volatility', tickfont=dict(size=18), title_font=dict(size=20))
fig.update_yaxes(title_text='', tickfont=dict(size=17))
fig.update_traces(cliponaxis=False)
fig.write_image('output/chart5_vpin_vol_all_tickers.png', scale=2)
fig.show()
json.dump({"caption":"Forward Volatility by VPIN Level, All 30 Stocks",
           "description":"Horizontal grouped bar chart comparing 5-day forward volatility on high-VPIN vs low-VPIN days across all 30 stocks in the universe."},
          open('output/chart5_vpin_vol_all_tickers.png.meta.json','w'))
print("done")

done


### Step 9 Results: The VPIN-Volatility Relationship Holds Almost Universally

**What the output shows:**
In the small-multiples grid, most of the 30 stocks show the VPIN line (red, dotted) fluctuating in a
pattern that visually tracks periods of sharper price movement (blue), though the relationship is
easier to spot in some tickers than others given the density of 30 panels. In the sorted bar chart,
the high-VPIN bar (red) sits at or above the low-VPIN bar (blue) for nearly every one of the 30
stocks, visually confirming the Step 3 result that 28 of 30 stocks showed a statistically significant
volatility difference.

**Interpreting the visuals:**
These charts introduce no new claims — they are visual confirmations of results already statistically
validated in Step 3. Their value lies in making an abstract statistical statement ("28 of 30 stocks,
p<0.05") immediately intuitive: a reviewer can scan the bar chart in seconds and see the consistent
red-above-blue pattern across nearly the entire universe, rather than parsing a 30-row table of
p-values one by one.

**Is this realistic enough to accept:**
Yes. Both charts faithfully reflect the same underlying numbers already computed and tested in
Step 3, with no selective framing or exaggeration — the 2 stocks that did not show significance are
still visible in both charts rather than being hidden or excluded, preserving the same intellectual
honesty maintained throughout the project.

**How this feeds into the next step:**
With the full set of visualizations now complete — the single-stock illustration (Step 8, Chart 1),
universe-wide validation (this step), strategy performance (Step 8, Chart 2), walk-forward robustness
(Step 8, Chart 3), and parameter sensitivity (Step 8, Chart 4) — all analytical and visual components
of the project are finished. The next and final step is assembling the complete research report in
Overleaf, weaving these charts and findings into a coherent, well-structured narrative for A Priori.

In [ ]:
print ('done!')

done!
